# 🎯 Object Detection with YOLO (Ultralytics) — No Fine-Tuning Needed!

**What you'll learn in this notebook:**
1. Install Ultralytics and load a **pretrained YOLO model** (YOLO11 — the latest & most powerful)
2. Run detection on **images from the internet**
3. Read the **bounding boxes** (coordinates, class names, confidence scores)
4. Use your **webcam** to detect objects in real time (Colab-style)

> 💡 **Key idea:** YOLO models from Ultralytics come pretrained on the **COCO dataset** (80 common object classes: person, car, dog, phone, ...). We can use them directly — zero training required!

## 1️⃣ Setup — Install Ultralytics

One line. That's it. This installs the `ultralytics` package which includes YOLO11, YOLOv8, and more.

In [ ]:
%pip install ultralytics -q

import ultralytics
ultralytics.checks()  # prints version, GPU availability, etc.

## 2️⃣ Load a Pretrained YOLO Model

Ultralytics model sizes (trade-off between speed 🏃 and accuracy 🎯):

| Model | Size | Speed | Accuracy (mAP) |
|-------|------|-------|----------------|
| `yolo11n.pt` | nano | ⚡ fastest | lowest |
| `yolo11s.pt` | small | fast | good |
| `yolo11m.pt` | medium | balanced | better |
| `yolo11l.pt` | large | slower | high |
| `yolo11x.pt` | **x-large** | slowest | 🏆 **highest** |

We'll use **`yolo11x.pt`** — the most powerful one. The weights download automatically on first use.

In [ ]:
from ultralytics import YOLO

# Load the biggest, most accurate YOLO11 model (pretrained on COCO)
model = YOLO("yolo11x.pt")

# What can it detect? 80 COCO classes:
print(f"Number of classes: {len(model.names)}")
print(model.names)

## 3️⃣ Detect Objects in Images from the Internet

You can pass a **URL directly** to the model — no manual download needed! 🔥

In [ ]:
# Detect on an image straight from a URL
results = model("https://ultralytics.com/images/bus.jpg")

In [ ]:
results

In [ ]:
# Show the result with boxes drawn
results[0].show()

In [ ]:
# Try more images — a street scene and people
urls = [
    "https://ultralytics.com/images/zidane.jpg",
    "https://images.pexels.com/photos/378570/pexels-photo-378570.jpeg?w=1200",  # busy street
]

for url in urls:
    res = model(url)
    res[0].show()

## 4️⃣ Extract the Bounding Boxes (the important part! 📦)

Each detection result has a `.boxes` attribute containing:
- **`xyxy`** → box corners `[x1, y1, x2, y2]` (top-left, bottom-right)
- **`conf`** → confidence score (0 to 1)
- **`cls`** → class index (use `model.names` to get the label)

In [ ]:
results = model("https://ultralytics.com/images/bus.jpg")
r = results[0]

print(f"{'Class':<15}{'Confidence':<12}{'Box [x1, y1, x2, y2]'}")
print("-" * 60)

for box in r.boxes:
    cls_id = int(box.cls[0])          # class index
    label  = model.names[cls_id]      # class name
    conf   = float(box.conf[0])       # confidence
    xyxy   = box.xyxy[0].tolist()     # coordinates

    print(f"{label:<15}{conf:<12.2f}{[round(v, 1) for v in xyxy]}")

### ⚙️ Useful inference options

You can control the detection behavior with arguments:

In [ ]:
# conf   = minimum confidence threshold (filter weak detections)
# classes = only detect specific classes (0 = person, 2 = car ...)
# imgsz  = inference image size (bigger = more accurate, slower)

results = model(
    "https://ultralytics.com/images/bus.jpg",
    conf=0.5,        # only keep detections above 50% confidence
    classes=[0],     # only detect "person"
    imgsz=1280,      # higher resolution inference
)
results[0].show()

## 5️⃣ Webcam Detection in Colab 📷

⚠️ **Important:** Colab runs on Google's servers — it can't access your webcam directly like `cv2.VideoCapture(0)` would on your laptop. Instead, we use a small **JavaScript bridge** that captures frames from your browser's camera and sends them to Python.
Live (near real-time) detection loop 🎥

This streams frames continuously from your browser camera to YOLO and displays the annotated output. Run the cell, allow camera access, and click **Stop** (interrupt the cell) when done.

> 💡 For smoother live FPS, you can swap `model` for a faster one: `model_fast = YOLO("yolo11n.pt")`

In [ ]:
from IPython.display import display, Javascript, Image as IPyImage
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2
import numpy as np
import PIL.Image
import io

# --- JavaScript video streaming bridge ---
def start_stream():
    js = Javascript('''
    let video = null;
    let stream = null;

    async function startStream() {
      video = document.createElement('video');
      video.style.display = 'block';
      stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
    }

    function captureFrame(quality) {
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      return canvas.toDataURL('image/jpeg', quality);
    }

    function stopStream() {
      if (stream) stream.getVideoTracks()[0].stop();
      if (video) video.remove();
    }
    ''')
    display(js)
    eval_js('startStream()')

def get_frame(quality=0.8):
    data = eval_js(f'captureFrame({quality})')
    binary = b64decode(data.split(',')[1])
    return cv2.imdecode(np.frombuffer(binary, np.uint8), cv2.IMREAD_COLOR)

def stop_stream():
    eval_js('stopStream()')

# --- Live detection loop ---
# Tip: use a faster model for live video
live_model = YOLO("yolo11n.pt")   # nano = fast; change to model (11x) for accuracy

start_stream()
display_handle = display(None, display_id=True)

try:
    while True:
        frame = get_frame()
        results = live_model(frame, verbose=False)
        annotated = results[0].plot()

        # Encode and update the displayed image in place
        _, buf = cv2.imencode('.jpg', annotated)
        display_handle.update(IPyImage(data=buf.tobytes()))
except KeyboardInterrupt:
    pass
finally:
    stop_stream()
    print("Stream stopped ✅")